<a href="https://colab.research.google.com/github/enedenao/tp_labo/blob/Jose/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP Grupal — Laboratorio de Métodos Cuantitativos (FCE UBA, 2C 2026)

**Integrantes:** Bautista Cortés · Gabriel Goldadler · José Luis Joannás · Naomi Asato · @dagut-rrd

**Repositorio:** https://github.com/enedenao/tp_labo

**Pregunta de investigación (versión actualizada):**

¿Qué personajes concentran el diálogo en Los Simpson, y existe una asociación entre el protagonismo de los personajes principales y la valoración de los episodios en IMDb que se mantenga a lo largo de las décadas?

## 0. Configuración del entorno

Importamos las bibliotecas que vamos a usar en todo el análisis:

- **pandas**: carga y manipulación del dataset (DataFrames, `groupby`, filtros).
- **numpy**: operaciones numéricas auxiliares.
- **matplotlib** y **seaborn**: gráficos. Seaborn se apoya en matplotlib y simplifica gráficos estadísticos (boxplots, barras agrupadas).
- **os**: crear la carpeta donde se guardan los gráficos exportados para el informe.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs("figuras", exist_ok=True)

> **Asistencia de IA:** Claude (Claude Project "tp labo"). Propuso el uso de os

## 1. Carga del dataset

**Dataset:** The Simpsons Data recopilación de Todd Schneider

**Fuente:** publicado en Kaggle

**Página del recurso:** https://github.com/sujanjoejacob/Text-mining-with-Simpsons-Data


Cargamos el CSV/ZIP directamente desde el repo, para que el notebook sea reproducible sin depender de archivos locales.

In [3]:
URL_EPISODIOS = "https://raw.githubusercontent.com/enedenao/tp_labo/main/data/simpsons_episodes.csv"
URL_DIALOGOS = "https://raw.githubusercontent.com/enedenao/tp_labo/main/data/simpsons_script_lines.csv.zip"

episodios = pd.read_csv(URL_EPISODIOS)
dialogos = pd.read_csv(URL_DIALOGOS, on_bad_lines="skip", low_memory=False)

print(f"Episodios: {episodios.shape[0]:,} filas | {episodios.shape[1]} columnas")
print(f"Dialogos:   {dialogos.shape[0]:,} filas | {dialogos.shape[1]} columnas")

Episodios: 600 filas | 13 columnas
Dialogos:   158,248 filas | 13 columnas


**Decisión de limpieza en la carga:** algunas filas del archivo de diálogos están mal armadas: tienen comas de más dentro del texto, y pandas no puede separar bien las columnas. Por defecto, eso hace que la carga falle. Con `on_bad_lines="skip"` se descartan esas filas: son 23 sobre 158.271 (menos del 0,02%), por lo que no afectan el análisis.

> **Asistencia de IA:** Claude (Claude Project "tp labo"). Se le compartió el CSV de diálogos y detectó las filas mal formadas; propuso `on_bad_lines="skip"`.

## 3.1 Exploración inicial

### a) Primeras y últimas filas

Mostramos el inicio y el final de cada tabla para ver cómo vienen los datos y detectar problemas a simple vista.

In [4]:
episodios.head()

,id,title,original_air_date,production_code,season,number_in_season,number_in_series,us_viewers_in_millions,views,imdb_rating,imdb_votes,image_url,video_url
0,10,Homer's Night Out,1990-03-25,7G10,1,10,10,30.3,50816.0,7.4,1511.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/275197507879
1,12,Krusty Gets Busted,1990-04-29,7G12,1,12,12,30.4,62561.0,8.3,1716.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/288019523914
2,14,"Bart Gets an ""F""",1990-10-11,7F03,2,1,14,33.6,59575.0,8.2,1638.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/260539459671
3,17,Two Cars in Every Garage and Three Eyes on Eve...,1990-11-01,7F01,2,4,17,26.1,64959.0,8.1,1457.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/260537411822
4,19,Dead Putting Society,1990-11-15,7F08,2,6,19,25.4,50691.0,8.0,1366.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/260539459670


In [5]:
episodios.tail()

,id,title,original_air_date,production_code,season,number_in_season,number_in_series,us_viewers_in_millions,views,imdb_rating,imdb_votes,image_url,video_url
595,500,At Long Last Leave,2012-02-19,PABF07,23,14,500,5.77,45226.0,7.0,580.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/230541891832
596,502,How I Wet Your Mother,2012-03-11,PABF08,23,16,502,4.97,44818.0,7.3,541.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/230597187582
597,511,Adventures in Baby-Getting,2012-11-04,PABF18,24,3,511,5.65,39687.0,6.9,460.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/221688899646
598,542,Diggs,2014-03-09,SABF08,25,12,542,2.69,39292.0,6.4,473.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/310529091639
599,548,Days of Future Future,2014-04-13,SABF13,25,18,548,3.64,55742.0,7.0,504.0,http://static-media.fxx.com/img/FX_Networks_-_...,http://www.simpsonsworld.com/video/310257731892


In [6]:
dialogos.head()

,id,episode_id,number,raw_text,timestamp_in_ms,speaking_line,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
0,9549,32,209,"Miss Hoover: No, actually, it was a little of ...",848000,true,464,3.0,Miss Hoover,Springfield Elementary School,"No, actually, it was a little of both. Sometim...",no actually it was a little of both sometimes ...,31.0
1,9550,32,210,Lisa Simpson: (NEAR TEARS) Where's Mr. Bergstrom?,856000,true,9,3.0,Lisa Simpson,Springfield Elementary School,Where's Mr. Bergstrom?,wheres mr bergstrom,3.0
2,9551,32,211,Miss Hoover: I don't know. Although I'd sure l...,856000,true,464,3.0,Miss Hoover,Springfield Elementary School,I don't know. Although I'd sure like to talk t...,i dont know although id sure like to talk to h...,22.0
3,9552,32,212,Lisa Simpson: That life is worth living.,864000,true,9,3.0,Lisa Simpson,Springfield Elementary School,That life is worth living.,that life is worth living,5.0
4,9553,32,213,Edna Krabappel-Flanders: The polls will be ope...,864000,true,40,3.0,Edna Krabappel-Flanders,Springfield Elementary School,The polls will be open from now until the end ...,the polls will be open from now until the end ...,33.0


In [7]:
dialogos.tail()

,id,episode_id,number,raw_text,timestamp_in_ms,speaking_line,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
158243,9544,32,204,Miss Hoover: (OFF LISA'S REACTION) I'm back.,831000,true,464,3.0,Miss Hoover,Springfield Elementary School,I'm back.,im back,2.0
158244,9545,32,205,"Miss Hoover: You see, class, my Lyme disease t...",839000,true,464,3.0,Miss Hoover,Springfield Elementary School,"You see, class, my Lyme disease turned out to ...",you see class my lyme disease turned out to be,10.0
158245,9546,32,206,Miss Hoover: Psy-cho-so-ma-tic.,842000,true,464,3.0,Miss Hoover,Springfield Elementary School,Psy-cho-so-ma-tic.,psy-cho-so-ma-tic,1.0
158246,9547,32,207,Ralph Wiggum: Does that mean you were crazy?,844000,true,119,3.0,Ralph Wiggum,Springfield Elementary School,Does that mean you were crazy?,does that mean you were crazy,6.0
158247,9548,32,208,"JANEY: No, that means she was faking it.",845000,true,4,3.0,JANEY,Springfield Elementary School,"No, that means she was faking it.",no that means she was faking it,7.0


### b) Estructura: dimensiones, columnas, tipos de dato y cobertura

`.dtypes` muestra el tipo de dato de cada columna. Una columna que debería ser numérica pero aparece como texto (`object`) suele indicar datos sucios.

In [8]:
print(episodios.dtypes)

id                          int64
title                      object
original_air_date          object
production_code            object
season                      int64
number_in_season            int64
number_in_series            int64
us_viewers_in_millions    float64
views                     float64
imdb_rating               float64
imdb_votes                float64
image_url                  object
video_url                  object
dtype: object


In [9]:
print(dialogos.dtypes)

id                      int64
episode_id              int64
number                  int64
raw_text               object
timestamp_in_ms        object
speaking_line          object
character_id           object
location_id           float64
raw_character_text     object
raw_location_text      object
spoken_words           object
normalized_text        object
word_count            float64
dtype: object


In [10]:
print("Fecha de emisión: de", episodios["original_air_date"].min(), "a", episodios["original_air_date"].max())
print("Temporadas:", episodios["season"].nunique())
print("Episodios en la tabla de episodios:", episodios["id"].nunique())                    #nunique() cuenta cuántos valores distintos hay en una columna.
print("Episodios con líneas de diálogo:", dialogos["episode_id"].nunique())

Fecha de emisión: de 1989-12-17 a 2016-10-16
Temporadas: 28
Episodios en la tabla de episodios: 600
Episodios con líneas de diálogo: 564


Hay 36 episodios sin dialogo

### c) Calidad de los datos: nulos, duplicados y estadísticas descriptivas

Contamos valores faltantes y filas repetidas, y resumimos las columnas numéricas. Revisamos además los valores de `speaking_line`, que indica si una línea es diálogo o una acotación del guion.

In [11]:
print(episodios.isnull().sum()) #tenemos 3 episodios (imdb_rating) sin datos que representa el 0.5%
print("Filas duplicadas:", episodios.duplicated().sum()) #no hay duplicados

id                        0
title                     0
original_air_date         0
production_code           0
season                    0
number_in_season          0
number_in_series          0
us_viewers_in_millions    6
views                     4
imdb_rating               3
imdb_votes                3
image_url                 4
video_url                 4
dtype: int64
Filas duplicadas: 0


In [12]:
print(dialogos.isnull().sum()) #el exceso de nulos corresponden a acotaciones del guion
print("Filas duplicadas:", dialogos.duplicated().sum()) #no tenemos filas duplicadas

id                        0
episode_id                0
number                    0
raw_text                  0
timestamp_in_ms           0
speaking_line             0
character_id          17521
location_id             407
raw_character_text    17522
raw_location_text       408
spoken_words          26159
normalized_text       26184
word_count            26159
dtype: int64
Filas duplicadas: 0


In [13]:
episodios[["imdb_rating", "us_viewers_in_millions", "imdb_votes"]].describe()

,imdb_rating,us_viewers_in_millions,imdb_votes
count,597.000000,594.000000,597.000000
mean,7.386097,11.842795,832.365159
std,0.732439,6.414218,396.117937
min,4.500000,2.320000,104.000000
25%,6.900000,7.055000,560.000000
50%,7.300000,10.300000,697.000000
75%,8.000000,15.250000,1095.000000
max,9.200000,33.600000,3734.000000


**Rating**:

Promedio 7,4, mediana 7,3, desvío 0,73.

El 50% de los episodios está entre 6,9 y 8,0 (cuartiles 25% y 75%).

El mínimo, 4,5, está muy por debajo del resto: es un episodio atípico.

Que el rango sea estrecho importa para la pregunta: las diferencias de rating entre episodios son chicas, así que cualquier asociación con el protagonismo probablemente sea moderada.

**Audiencia**:

Promedio 11,8 millones, pero mediana 10,3. Cuando el promedio es mayor que la mediana, hay valores altos que “tiran” el promedio para arriba. El rango va de 2,3 a 33,6 millones, lo que sugiere una caída importante de la audiencia a lo largo del tiempo, a verificar en el análisis de gráfico.

In [14]:
dialogos["word_count"].describe()
#las 132.089 lineas con palabras registradas tienen en promedio casi 10 palabras, con una mediana de 8: predominan los dialogos breves

,word_count
count,132089.000000
mean,9.887175
std,8.055425
min,0.000000
25%,4.000000
50%,8.000000
75%,13.000000
max,122.000000


In [15]:
dialogos["speaking_line"].value_counts(dropna=False) #132.089 lineas habladas y 26.158 acotaciones
#hay una fila que es dialogo y quedo en columna equivocada

,count
speaking_line,
true,132089
false,26158
Guess what. I also play Frankenstein!,1


In [16]:
episodios["season"].value_counts().sort_index() #temporada 28 incompleta

,count
season,
1,13
2,22
3,24
4,22
5,22
6,25
7,25
8,25
9,25


### Hallazgos de la exploración

**Episodios**

- 600 episodios de 28 temporadas, emitidos entre el 17/12/1989 y el 16/10/2016. La mayoría de las temporadas tiene entre 20 y 25 episodios; la temporada 1 tiene 13, por ser más corta, y la **temporada 28 tiene solo 4**, (datos se recopilaron poco después del estreno de la 28°temporada).
- No hay filas duplicadas y los `id` van del 1 al 600 sin repetirse.
- Hay pocos nulos: 6 en audiencia (`us_viewers_in_millions`), 4 en `views` y 3 en `imdb_rating` e `imdb_votes`. Son menos del 1% de los episodios.
- `imdb_rating` va de 4,5 a 9,2, con promedio de 7,4 y desvío de 0,73: la mayoría de los episodios se concentra en un rango estrecho.
- La audiencia en EE.UU. va de 2,3 a 33,6 millones de espectadores: hay una diferencia muy grande entre los episodios más y menos vistos, que habrá que analizar en el tiempo.
- `original_air_date` está como texto (`object`), no como fecha, y no hay una columna de año. Para analizar por décadas hay que convertirla y crear esa columna.
- Las filas no están ordenadas cronológicamente (el `head()` arranca en el episodio 10 y el `tail()` termina en el 548).

**Diálogos**

- 158.248 líneas de guion (se descartaron 23 filas mal formadas al cargar), sin filas duplicadas.
- Solo 564 de los 600 episodios tienen líneas de diálogo: 36 episodios no tienen guion y no pueden incluirse en el análisis de protagonismo. [Completar: si se concentran en alguna época.]
- `speaking_line` no es booleana: viene como texto (`"true"` / `"false"`). Hay 132.089 líneas habladas y 26.158 acotaciones, más una fila corrupta con una frase del guion en esa columna.
- `timestamp_in_ms` y `character_id` deberían ser numéricas pero aparecen como texto (`object`), por esa misma fila corrupta.
- Los nulos en `spoken_words`, `normalized_text` y `word_count` (unos 26.000) corresponden a las acotaciones del guion, que no tienen palabras dichas. Los nulos en `character_id` y `raw_character_text` corresponden mayormente a acotaciones sin personaje.
- Las líneas con palabras tienen en promedio casi 10 palabras (mediana 8, máximo 122). Hay líneas con 0 palabras registradas.
- Hay 6.763 nombres de personajes distintos, y un mismo personaje puede aparecer con varios nombres: por ejemplo, 63 nombres contienen "Bart" ("Adult Bart", "Teenage Bart"), aunque "Bart Simpson" concentra la gran mayoría de las líneas. La búsqueda por texto incluye además personajes que no son Bart, como "Bartender", por lo que una unificación automática no sería confiable.

**Para la etapa de transformaciones:**

- Filtrar solo las líneas habladas (`speaking_line == "true"`), lo que también descarta la fila corrupta.
- Convertir `original_air_date` a fecha y crear las columnas de año y década.
- Convertir `timestamp_in_ms` y `character_id` a numéricas.
- Decidir cómo tratar los nulos de rating y audiencia, la temporada 28 (incompleta) y las variantes de nombres de personajes.

>**Asistencia de IA:** Claude (Claude Project "tp labo"). Se usó como apoyo para para revisar y mejorar todo lo trabajado.

## 3.2 Pregunta de investigación

**a) ¿Cuál es la pregunta?**

¿Qué personajes concentran el diálogo en Los Simpson, y existe una asociación entre el protagonismo de los personajes principales y la valoración de los episodios en IMDb que se mantenga a lo largo de las décadas?

Para que sea respondible con los datos disponibles, los conceptos se miden así:
- **Concentración del diálogo:** cantidad de palabras habladas por personaje (`character_id`) sobre el total de palabras habladas.
- **Personajes principales:** los cuatro con más palabras habladas, definidos a partir del ranking de los propios datos.
- **Protagonismo de un episodio:** porcentaje de las palabras habladas del episodio que dicen los personajes principales (`pct_principales`).
- **Valoración:** calificación del episodio en IMDb (`imdb_rating`).
- **Décadas:** tres períodos según la fecha de emisión (1989-1999, 2000-2009 y 2010-2016). No todos cubren diez años: la serie se estrenó a fines de 1989 y el dataset termina en 2016.
- **Asociación:** relación estadística entre protagonismo y valoración (por ejemplo, correlación y comparación entre períodos). No implica causalidad.

**b) ¿Por qué vale la pena responderla?**

La serie lleva más de 25 años en el aire y su elenco es muy amplio, con una familia central y cientos de personajes secundarios. Responder la pregunta ayuda a entender si los episodios en los que la familia principal concentra más diálogo son mejor o peor valorados por el público, y si esa relación cambió a lo largo de la historia de la serie. Ilumina una decisión típica de la producción de contenido: cuánto peso darle al elenco central frente a los personajes secundarios al escribir los guiones.

**c) ¿Qué respuesta encontraron?**

[Completar al finalizar el análisis, con los hallazgos concretos: quiénes concentran el diálogo, el sentido y la magnitud de la asociación con el rating IMDb, y si se mantiene entre períodos.]


## 3.3 Transformaciones

### Transformación 1: limpieza de los diálogos y construcción del protagonismo por episodio
**Qué se hace:**
1. Se conservan solo las líneas habladas (`speaking_line == "true"`), se convierten `timestamp_in_ms` y `character_id` a numéricas y se eliminan las líneas sin personaje.
2. Se arma el ranking de personajes por cantidad de palabras habladas y se definen como "principales" los cuatro primeros.
3. Se calcula, para cada episodio, el porcentaje de las palabras habladas que dicen los principales (`pct_principales`).

**Por qué:**
- El protagonismo se mide con lo que los personajes dicen, no con las acotaciones del guion (26.158 líneas). Al filtrar por `"true"` también se descarta la fila corrupta que hacía que `timestamp_in_ms` y `character_id` quedaran como texto.
- Se agrupa por `character_id` y no por el nombre escrito, porque un mismo personaje puede aparecer con variantes de nombre; el id es único. Definir los principales con el ranking evita elegirlos a mano y responde la primera parte de la pregunta.
- Se miden **palabras** y no líneas (criterio acordado por el grupo), porque una línea corta y una larga no pesan igual: las palabras reflejan mejor cuánto "habla" cada personaje.
- Se usa un **porcentaje** y no una cantidad para que episodios con más o menos diálogo sean comparables entre sí.


In [17]:
# 1. Líneas habladas (descarta acotaciones y la fila corrupta)
print("Antes:", dialogos.shape)
dialogos_h = dialogos[dialogos["speaking_line"] == "true"].copy()
print("Líneas habladas:", dialogos_h.shape)   # esperado: 132.089

# 2. Tipos numéricos y líneas sin personaje
dialogos_h["timestamp_in_ms"] = pd.to_numeric(dialogos_h["timestamp_in_ms"], errors="coerce")
dialogos_h["character_id"] = pd.to_numeric(dialogos_h["character_id"], errors="coerce")
n_antes = len(dialogos_h)
dialogos_h = dialogos_h.dropna(subset=["character_id"]).copy()
dialogos_h["character_id"] = dialogos_h["character_id"].astype(int)
print("Líneas sin personaje eliminadas:", n_antes - len(dialogos_h))

# 3. Ranking de personajes por palabras
ranking = dialogos_h.groupby("character_id").agg(
    nombre=("raw_character_text", "first"),
    palabras=("word_count", "sum"))
ranking = ranking.sort_values("palabras", ascending=False)
ranking["pct_del_total"] = ranking["palabras"] / ranking["palabras"].sum() * 100
display(ranking.head(10))
ids_principales = ranking.head(4).index.tolist()

# 4. Protagonismo por episodio (en palabras)
dialogos_h["es_principal"] = dialogos_h["character_id"].isin(ids_principales)
dialogos_h["palabras_principales"] = dialogos_h["word_count"] * dialogos_h["es_principal"]
por_ep = dialogos_h.groupby("episode_id").agg(
    n_palabras=("word_count", "sum"),
    palabras_principales=("palabras_principales", "sum"))
por_ep["pct_principales"] = por_ep["palabras_principales"] / por_ep["n_palabras"] * 100
por_ep = por_ep.reset_index()
print(por_ep["n_palabras"].describe())

Antes: (158248, 13)
Líneas habladas: (132089, 13)
Líneas sin personaje eliminadas: 2


,nombre,palabras,pct_del_total
character_id,,,
2,Homer Simpson,273780.0,20.963937
1,Marge Simpson,125666.0,9.622522
8,Bart Simpson,110814.0,8.485272
9,Lisa Simpson,100576.0,7.701326
15,C. Montgomery Burns,36623.0,2.804304
17,Moe Szyslak,32855.0,2.515780
3,Seymour Skinner,28138.0,2.154589
11,Ned Flanders,22909.0,1.754193
139,Krusty the Clown,20897.0,1.600129


count     564.000000
mean     2315.526596
std       267.219824
min       761.000000
25%      2150.750000
50%      2333.000000
75%      2493.250000
max      3066.000000
Name: n_palabras, dtype: float64


### Transformación 2: fecha y década, unión con el protagonismo y base final
**Qué se hace:**
1. Se convierte `original_air_date` a fecha y se crean `anio` y `decada` (1989-1999, 2000-2009 y 2010-2016).
2. Se une `episodios` con el protagonismo por episodio y se marca con `tiene_guion` qué episodios tienen diálogo.
3. La base final para el análisis conserva solo los episodios con guion.

**Por qué:**
- Sin variable temporal no se puede responder si la asociación se mantiene a lo largo de las décadas.
- **Aclaración sobre las décadas:** no todos los grupos cubren diez años. 1989-1999 incorpora 1989 porque la serie se estrenó el 17/12/1989 y un grupo de un solo episodio no se puede comparar. 2000-2009 es la única década completa. 2010-2016 es incompleta (7 años) porque el dataset termina en octubre de 2016.
- **Episodios sin guion:** 36 de los 600 episodios no tienen líneas de diálogo en el dataset. Se marcan en lugar de borrarlos de `episodios`, porque tienen rating y audiencia y sirven para análisis que no dependen del guion; se excluyen solo de la base de protagonismo. La base final queda con **564 episodios**.
- **Los nulos de rating (3 episodios) y la temporada 28 (4 episodios, incompleta) ya estaban entre esos 36**, por lo que no hubo que excluir episodios por esos motivos: la base de 564 ya no los contiene. Así, no se rellenó ningún nulo con el promedio, lo que habría inventado datos.
-  **Limitación:** de los 36 episodios sin guion, 33 pertenecen a 2010-2016, de modo que ese período pierde el 21,9% de sus episodios (118 de 151 quedan en la base). Las conclusiones de ese período se apoyan en menos episodios y deben leerse con cautela. El dataset no permite saber por qué falta el guion de esos episodios, y no se puede asegurar que la falta sea aleatoria.



In [18]:
# 1. Fecha, año y década
episodios["original_air_date"] = pd.to_datetime(episodios["original_air_date"], errors="coerce")
print("Fechas inválidas:", episodios["original_air_date"].isnull().sum())
episodios["anio"] = episodios["original_air_date"].dt.year

def asignar_decada(anio):
    if anio <= 1999:
        return "1989-1999"
    elif anio <= 2009:
        return "2000-2009"
    else:
        return "2010-2016"

episodios["decada"] = episodios["anio"].apply(asignar_decada)
print(episodios.groupby("decada").size())

# 2. Unión con el protagonismo y marca de episodios sin guion
analisis = episodios.merge(por_ep, left_on="id", right_on="episode_id", how="left")
print(len(episodios), len(analisis))   # deben ser iguales (600 y 600)
analisis["tiene_guion"] = analisis["n_palabras"].notna()
print(analisis["tiene_guion"].value_counts())   # 564 True / 36 False

# 3. ¿Qué porcentaje de cada período no tiene guion?
sin_guion = analisis[~analisis["tiene_guion"]]
total_por_decada = analisis.groupby("decada").size()
faltantes_por_decada = sin_guion.groupby("decada").size()
print(faltantes_por_decada)
print((faltantes_por_decada / total_por_decada * 100).round(1))

# 4. Base final y controles
base = analisis[analisis["tiene_guion"]].copy()
print("Episodios en la base final:", len(base))                 # 564
print("Episodios sin rating en la base:", base["imdb_rating"].isnull().sum())   # debe dar 0
print("Episodios de la temporada 28 en la base:", (base["season"] == 28).sum()) # debe dar 0
print(base.groupby("decada").size())

Fechas inválidas: 0
decada
1989-1999    235
2000-2009    214
2010-2016    151
dtype: int64
600 600
tiene_guion
True     564
False     36
Name: count, dtype: int64
decada
2000-2009     3
2010-2016    33
dtype: int64
decada
1989-1999     NaN
2000-2009     1.4
2010-2016    21.9
dtype: float64
Episodios en la base final: 564
Episodios sin rating en la base: 0
Episodios de la temporada 28 en la base: 0
decada
1989-1999    235
2000-2009    211
2010-2016    118
dtype: int64


### Asistencia de IA (Parte 1, etapa 2)
**Herramienta:** Claude

**Celdas con asistencia:** pregunta de investigación (3.2) y transformaciones 1 y 2 (3.3).

**Tipo de ayuda:** aportó ideas y código.

**Prompt utilizado**  « Quiero que te comportes como un analista de datos. El tp dejó una lista de transformaciones para esta etapa, pero quiero que agregues más cosas.» Se adjuntaron la lista de transformaciones, el notebook con la exploración inicial y las pautas del TP.

## 3.3 Resumen estadístico agrupado con `.groupby()`

**Idea general (en palabras simples):** `.groupby()` divide la tabla en "grupos" (por ejemplo, un grupo por década), calcula un número resumen en cada grupo (promedio, suma, cantidad) y nos permite comparar los grupos entre sí. Es como separar los episodios en tres pilas según su época y sacar el promedio de cada pila.

### Variables que usamos y por qué

| Variable | Tabla | Qué representa | Por qué la usamos |
|---|---|---|---|
| `character_id` | diálogos | Identificador único del personaje | Evita errores por nombres escritos distinto ("Bart", "Adult Bart") |
| `word_count` | diálogos | Palabras de cada línea | Mide cuánto "habla" un personaje (una línea larga pesa más que una corta) |
| `episode_id` | diálogos | Episodio al que pertenece la línea | Permite sumar el diálogo por episodio y unir con la tabla de episodios |
| `pct_principales` | construida | % de palabras del episodio dichas por Homer, Marge, Bart y Lisa | Es nuestra medida de **protagonismo** y es comparable entre episodios |
| `imdb_rating` | episodios | Nota promedio del episodio en IMDb (0–10) | Es nuestra medida de **valoración** |
| `decada` | construida | Período de emisión (3 grupos) | Permite ver si la relación cambia con el tiempo |

### Estadístico agrupado 1: ¿Quién concentra el diálogo? (por personaje y por década)

**Qué hacemos:** agrupamos las palabras habladas por personaje y por década, y calculamos qué porcentaje del total de palabras de cada década dice cada uno de los cuatro principales.

**Por qué:** responde la primera parte de la pregunta (*¿qué personajes concentran el diálogo?*) y además muestra si el reparto cambió con el tiempo. Usamos **porcentajes** y no palabras absolutas porque las décadas tienen distinta cantidad de episodios.

In [19]:
# Nombre de cada personaje principal (tomado del ranking ya calculado)
dh = dialogos_h.merge(episodios[["id", "decada"]], left_on="episode_id", right_on="id", how="left", suffixes=("", "_ep"))
dh["nombre"] = dh["character_id"].map(ranking["nombre"])

# Total de palabras habladas por década (el "100%" de cada época)
palabras_decada = dh.groupby("decada")["word_count"].sum()

# Palabras de cada principal por década -> tabla ancha (filas = década, columnas = personaje)
pal_principales = (dh[dh["es_principal"]]
                   .groupby(["decada", "nombre"])["word_count"].sum()
                   .unstack("nombre"))

# Pasamos a porcentaje del total de la década
pct_decada = pal_principales.div(palabras_decada, axis=0) * 100
pct_decada["Total 4 principales"] = pct_decada.sum(axis=1)
pct_decada = pct_decada.round(1)
display(pct_decada)

# Para comparar: porcentaje de toda la serie
print("Porcentaje sobre toda la serie:")
print((ranking.head(4)["pct_del_total"]).round(1).to_string())
print("Total 4 principales:", ranking.head(4)["pct_del_total"].sum().round(1), "%")

nombre,Bart Simpson,Homer Simpson,Lisa Simpson,Marge Simpson,Total 4 principales
decada,,,,,
1989-1999,9.1,21.2,7.3,9.3,46.9
2000-2009,7.8,21.2,7.9,10.4,47.3
2010-2016,8.5,19.9,8.2,9.0,45.6


Porcentaje sobre toda la serie:
character_id
2    21.0
1     9.6
8     8.5
9     7.7
Total 4 principales: 46.8 %


**Cómo leer este resultado (completar con los números reales):**
- Cada fila es una época y cada número es "de cada 100 palabras dichas en esa época, cuántas dice ese personaje".
- Homer encabeza el ranking con cerca del 21% de todas las palabras de la serie, y los cuatro principales juntos rondan el 47%: **casi la mitad del diálogo está en solo cuatro personajes**, de entre miles.
- Fijarse en la columna *Total 4 principales*: si **baja** de una década a otra, el diálogo se fue repartiendo hacia personajes secundarios; si **sube**, la familia ganó protagonismo.
- Completar: `[ej.: "El peso de los 4 principales pasó de X% en 1989-1999 a Y% en 2010-2016"]`.

**Cómo leer este resultado (completar con los números reales):**
- Cada fila es una época y cada número es "de cada 100 palabras dichas en esa época, cuántas dice ese personaje".
- Homer encabeza el ranking con cerca del 21% de todas las palabras de la serie, y los cuatro principales juntos rondan el 47%: **casi la mitad del diálogo está en solo cuatro personajes**, de entre miles.
- Fijarse en la columna *Total 4 principales*: si **baja** de una década a otra, el diálogo se fue repartiendo hacia personajes secundarios; si **sube**, la familia ganó protagonismo.
- Completar: `[ej.: "El peso de los 4 principales pasó de X% en 1989-1999 a Y% en 2010-2016"]`.

In [20]:
resumen_decada = base.groupby("decada").agg(
    episodios=("id", "count"),
    protagonismo_prom=("pct_principales", "mean"),
    protagonismo_mediana=("pct_principales", "median"),
    protagonismo_desvio=("pct_principales", "std"),
    rating_prom=("imdb_rating", "mean"),
    rating_mediana=("imdb_rating", "median"),
    rating_desvio=("imdb_rating", "std"),
).round(2)
display(resumen_decada)

,episodios,protagonismo_prom,protagonismo_mediana,protagonismo_desvio,rating_prom,rating_mediana,rating_desvio
decada,,,,,,,
1989-1999,235,47.14,48.20,13.52,8.07,8.1,0.52
2000-2009,211,47.25,47.77,12.19,7.05,7.1,0.40
2010-2016,118,45.82,47.05,13.19,6.79,6.9,0.47


**Cómo leer este resultado (completar con los números reales):**
- `protagonismo_prom`: porcentaje típico de palabras de un episodio dicho por Homer, Marge, Bart y Lisa.
- `rating_prom`: nota típica de un episodio en IMDb. Si baja década tras década mientras el protagonismo se mantiene o sube, ya vemos una señal de que **no es la familia lo que explica la caída de la nota** (otro factor, como el desgaste de la serie, estaría pesando más).
- `desvío`: qué tanto varían los episodios dentro de la década. Un desvío chico indica episodios muy parecidos entre sí.
- Aviso: la década 2010-2016 tiene menos episodios (118) por la falta de guiones; sus promedios son menos firmes.

### Estadístico agrupado 3: valoración según nivel de protagonismo (cuartiles)

**Qué hacemos:** dividimos los episodios en 4 grupos de igual tamaño según su protagonismo (cuartil 1 = episodios con **menos** protagonismo de la familia; cuartil 4 = con **más**) y calculamos el rating promedio de cada grupo. Lo hacemos para toda la serie y por década.

**Por qué:** es la forma más intuitiva de ver la asociación: si al pasar de "poco protagonismo" a "mucho protagonismo" el rating sube o baja de forma ordenada, hay una relación. Sirve además como respaldo del diagrama de dispersión.

In [21]:
etiquetas = ["Q1 (menos)", "Q2", "Q3", "Q4 (más)"]

# Cuartiles sobre toda la serie
base["nivel_protagonismo"] = pd.qcut(base["pct_principales"], q=4, labels=etiquetas)
print("Rating promedio según nivel de protagonismo (toda la serie):")
display(base.groupby("nivel_protagonismo", observed=True)["imdb_rating"]
        .agg(episodios="count", rating_prom="mean", rating_mediana="median").round(2))

# Lo mismo, separado por década
print("Rating promedio según nivel de protagonismo y década:")
cuartiles_decada = (base.groupby(["decada", "nivel_protagonismo"], observed=True)["imdb_rating"]
                    .mean().unstack("nivel_protagonismo").round(2))
display(cuartiles_decada)

Rating promedio según nivel de protagonismo (toda la serie):


,episodios,rating_prom,rating_mediana
nivel_protagonismo,,,
Q1 (menos),141,7.43,7.3
Q2,141,7.38,7.2
Q3,141,7.42,7.3
Q4 (más),141,7.46,7.3


Rating promedio según nivel de protagonismo y década:


nivel_protagonismo,Q1 (menos),Q2,Q3,Q4 (más)
decada,,,,
1989-1999,8.12,8.16,8.04,8.00
2000-2009,6.98,7.10,7.13,6.95
2010-2016,6.75,6.66,6.79,7.00


**Cómo leer este resultado (completar con los números reales):**
- Recorrer la fila de izquierda a derecha (de Q1 a Q4). Si los números **suben**, a más protagonismo mejor nota; si **bajan**, al revés; si **no muestran orden**, no hay relación clara.
- Diferencias de pocas décimas son pequeñas: recordar que el 50% central de los episodios está entre 6,9 y 8,0.
- Comparar las tres filas: si el patrón se repite en las tres décadas, la asociación **se mantiene**; si cambia de signo, **no se mantiene**.

### Estadístico complementario: correlación por década

**Qué hacemos:** calculamos la **correlación** entre protagonismo y rating en cada década. La correlación es un número entre -1 y 1: cerca de 0 significa "no hay relación"; positivo, "cuando uno sube el otro tiende a subir"; negativo, "cuando uno sube el otro tiende a bajar". Usamos Pearson (relación lineal) y Spearman (por ranking, menos sensible a casos extremos).

**Por qué:** convierte la impresión visual en un número y responde directamente si la asociación se mantiene entre períodos. **Correlación no es causalidad:** que dos cosas vayan juntas no prueba que una cause la otra.

In [22]:
def correlaciones(df):
    return pd.Series({
        "episodios": len(df),
        "pearson": df["pct_principales"].corr(df["imdb_rating"], method="pearson"),
        "spearman": df["pct_principales"].corr(df["imdb_rating"], method="spearman"),
    })

corr_decada = base.groupby("decada")[["pct_principales", "imdb_rating"]].apply(correlaciones)
corr_decada.loc["Toda la serie"] = correlaciones(base)
corr_decada["episodios"] = corr_decada["episodios"].astype(int)
display(corr_decada.round(2))

,episodios,pearson,spearman
decada,,,
1989-1999,235,-0.12,-0.12
2000-2009,211,0.00,-0.04
2010-2016,118,0.24,0.23
Toda la serie,564,0.01,0.02


**Cómo leer este resultado (completar con los números reales):**
- Regla práctica: entre -0,1 y 0,1 ≈ sin relación; hasta ±0,3 ≈ débil; ±0,3 a ±0,5 ≈ moderada; más de ±0,5 ≈ fuerte.
- Si Pearson y Spearman dan parecido, el resultado es robusto.
- Con pocos episodios (como 2010-2016) una correlación pequeña puede deberse al azar. Con ~100 episodios, valores menores a ±0,2 en general no son distinguibles del azar.
- Conclusión a redactar: `[¿hay asociación? ¿en qué sentido? ¿qué tan fuerte? ¿se repite en las tres décadas?]`

> **Asistencia de IA:** Claude (Claude Project "tp labo"). Se usó para proponer los estadísticos agrupados y redactar las explicaciones. **Prompt:** «Actuá como un analista de datos senior [...] identificá las variables más relevantes para la pregunta de investigación y calculá al menos dos estadísticos de resumen agrupados con .groupby(), con explicación para personas sin conocimiento previo.»

**Cómo leer este resultado (completar con los números reales):**
- Regla práctica: entre -0,1 y 0,1 ≈ sin relación; hasta ±0,3 ≈ débil; ±0,3 a ±0,5 ≈ moderada; más de ±0,5 ≈ fuerte.
- Si Pearson y Spearman dan parecido, el resultado es robusto.
- Con pocos episodios (como 2010-2016) una correlación pequeña puede deberse al azar. Con ~100 episodios, valores menores a ±0,2 en general no son distinguibles del azar.
- Conclusión a redactar: `[¿hay asociación? ¿en qué sentido? ¿qué tan fuerte? ¿se repite en las tres décadas?]`

> **Asistencia de IA:** Claude (Claude Project "tp labo"). Se usó para proponer los estadísticos agrupados y redactar las explicaciones. **Prompt:** «Actuá como un analista de datos senior [...] identificá las variables más relevantes para la pregunta de investigación y calculá al menos dos estadísticos de resumen agrupados con .groupby(), con explicación para personas sin conocimiento previo.»